In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
DATA_PATH = "book_bestseller_clean.csv"

df_books = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig",
)

print("데이터 크기:", df_books.shape)
print("컬럼:", df_books.columns.tolist())
print("상품명 결측치:", df_books["상품명"].isna().sum())

df_books[["상품명"]].head(10)

데이터 크기: (199, 8)
컬럼: ['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']
상품명 결측치: 0


,상품명
0,소년이 온다
1,모순
2,결국 국민이 합니다
3,혼모노
4,급류
5,초역 부처의 말
6,청춘의 독서(특별증보판)
7,어른의 행복은 조용하다
8,채식주의자
9,단 한 번의 삶(강물에디션 활판인쇄 한정판)


In [4]:
df_reco = df_books.copy()

df_reco["상품명"] = (

    df_reco["상품명"]

    .fillna("")

    .astype(str)

    .str.strip()

)

df_reco = (

    df_reco[df_reco["상품명"] != ""]

    .reset_index(drop=True)

)

print("추천에 사용할 도서 수:", len(df_reco))

추천에 사용할 도서 수: 199


### 실습 4. 콘텐츠 기반 추천 이해하기

In [6]:

vectors = np.array([

    [1, 1],

    [2, 2],

    [1, 0],

])


cosine_similarity(vectors)

array([[1.        , 1.        , 0.70710678],
       [1.        , 1.        , 0.70710678],
       [0.70710678, 0.70710678, 1.        ]])

### 실습 6. 실제 도서 제목을 TF-IDF로 변환하기

In [7]:
titles = df_reco["상품명"]

tfidf = TfidfVectorizer()

tfidf_matrix = tfidf.fit_transform(titles)

print("도서 수:", tfidf_matrix.shape[0])

print("단어 수:", tfidf_matrix.shape[1])

도서 수: 199
단어 수: 536


### 실습 7. 기준 도서 선택하기

In [14]:
selected_index = 134

selected_title = df_reco.loc[

    selected_index,

    "상품명",

]

print("선택 도서:", selected_title)

선택 도서: 지적 대화를 위한 넓고 얕은 지식 무한


### 실습 8. 선택 도서와 전체 도서의 유사도 계산하기

In [15]:
selected_vector = tfidf_matrix[selected_index]

In [16]:
similarity_scores = cosine_similarity(

    selected_vector,

    tfidf_matrix,

).flatten()

print("유사도 개수:", len(similarity_scores))

print("전체 도서 수:", len(df_reco))

유사도 개수: 199
전체 도서 수: 199


### 실습 9. 유사도가 높은 순서 확인하기

In [17]:

score_df = pd.DataFrame({

    "index": np.arange(len(df_reco)),

    "상품명": df_reco["상품명"],

    "similarity": similarity_scores,

})

score_df.sort_values(

    "similarity",

    ascending=False,

).head(10)

,index,상품명,similarity
134,134,지적 대화를 위한 넓고 얕은 지식 무한,1.000000
167,167,지적 대화를 위한 넓고 얕은 지식 1,0.910836
62,62,황현필의 진보를 위한 역사,0.123823
181,181,살아갈 날들을 위한 괴테의 시,0.123823
80,80,하루 한 장 나의 어휘력을 위한 필사 노트,0.102595
137,137,더 좋은 문장을 쓰고 싶은 당신을 위한 필사책,0.097872
37,37,더 나은 어휘를 쓰고 싶은 당신을 위한 필사책,0.096699
6,6,청춘의 독서(특별증보판),0.000000
5,5,초역 부처의 말,0.000000
9,9,단 한 번의 삶(강물에디션 활판인쇄 한정판),0.000000


### 실습 10. 자기 자신을 제외하고 Top 5 만들기

In [18]:
sorted_indices = similarity_scores.argsort()[::-1]

In [20]:
recommended_indices = [

    idx

    for idx in sorted_indices

        if idx != selected_index

][:5]

recommended_indices

[np.int64(167), np.int64(181), np.int64(62), np.int64(80), np.int64(137)]

In [21]:

result = df_reco.loc[

    recommended_indices,

    ["상품명"],

].copy()

result["similarity"] = [

    round(float(similarity_scores[idx]), 4)

    for idx in recommended_indices

]

result

,상품명,similarity
167,지적 대화를 위한 넓고 얕은 지식 1,0.9108
181,살아갈 날들을 위한 괴테의 시,0.1238
62,황현필의 진보를 위한 역사,0.1238
80,하루 한 장 나의 어휘력을 위한 필사 노트,0.1026
137,더 좋은 문장을 쓰고 싶은 당신을 위한 필사책,0.0979


In [22]:
available_columns = [

    column

    for column in [

        "상품명",

        "인물",

        "출판사",

        "분야",

    ]

    if column in df_reco.columns

]

result = df_reco.loc[

    recommended_indices,

    available_columns,

].copy()

result["similarity"] = [

    round(float(similarity_scores[idx]), 4)

    for idx in recommended_indices

]

result

,상품명,출판사,분야,similarity
167,지적 대화를 위한 넓고 얕은 지식 1,웨일북(whalebooks),인문,0.9108
181,살아갈 날들을 위한 괴테의 시,퍼스트펭귄,자기계발,0.1238
62,황현필의 진보를 위한 역사,역바연,역사/문화,0.1238
80,하루 한 장 나의 어휘력을 위한 필사 노트,위즈덤하우스,인문,0.1026
137,더 좋은 문장을 쓰고 싶은 당신을 위한 필사책,빅피시,인문,0.0979


In [23]:
def recommend_books(

    selected_index,

    df,

    tfidf_matrix,

    top_n=5,

):

    if selected_index not in df.index:

        raise IndexError(

            f"유효하지 않은 index입니다: {selected_index}"

        )

    selected_title = df.loc[

        selected_index,

        "상품명",

    ]

    similarity_scores = cosine_similarity(

        tfidf_matrix[selected_index],

        tfidf_matrix,

    ).flatten()

    sorted_indices = similarity_scores.argsort()[::-1]

    recommended_indices = []

    for idx in sorted_indices:

        # 선택한 자기 자신 제외

        if idx == selected_index:

            continue

        # 동일한 제목도 제외

        if df.loc[idx, "상품명"] == selected_title:

            continue

        recommended_indices.append(idx)

        if len(recommended_indices) >= top_n:

            break

    columns = [

        column

        for column in [

            "상품명",

            "인물",

            "출판사",

            "분야",

        ]

        if column in df.columns

    ]

    result = df.loc[

        recommended_indices,

        columns,

    ].copy()

    result["similarity"] = [

        round(float(similarity_scores[idx]), 4)

        for idx in recommended_indices

    ]

    return result.reset_index(drop=True)

In [30]:
selected_index = 134

print(

    "선택 도서:",

    df_reco.loc[selected_index, "상품명"],

)

recommendations = recommend_books(

    selected_index=selected_index,

    df=df_reco,

    tfidf_matrix=tfidf_matrix,

    top_n=5,

)

recommendations

선택 도서: 지적 대화를 위한 넓고 얕은 지식 무한


,상품명,출판사,분야,similarity
0,지적 대화를 위한 넓고 얕은 지식 1,웨일북(whalebooks),인문,0.9108
1,살아갈 날들을 위한 괴테의 시,퍼스트펭귄,자기계발,0.1238
2,황현필의 진보를 위한 역사,역바연,역사/문화,0.1238
3,하루 한 장 나의 어휘력을 위한 필사 노트,위즈덤하우스,인문,0.1026
4,더 좋은 문장을 쓰고 싶은 당신을 위한 필사책,빅피시,인문,0.0979


In [31]:
recommendations.to_csv(

    "chapter05_recommendations.csv",

    index=False,

    encoding="utf-8-sig",

)

### 실습 16. Chapter 06 Streamlit 연결 준비하기

In [ ]:
book_options = {

    f"{idx} | {row['상품명']}": idx

    for idx, row in df_reco.iterrows()

}